In [ ]:
# Scenario: Corporate Product Launch Broadcast
# Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
# “Product X launch in Q3, target market North America, budget $5M.”


# 📢 Coordinator Agent (Broadcaster)
# - Sends the launch announcement to all departments simultaneously.
# - This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

# 📈 Marketing Agent
# - Receives the broadcast.
# - Responds with a marketing strategy: campaigns, channels, and positioning.
# 💰 Finance Agent
# - Receives the broadcast.
# - Responds with budget allocation and ROI forecasts.
# 🏭 Operations Agent
# - Receives the broadcast.
# - Responds with production and supply chain actions.
# 👩‍⚖️ Legal Agent
# - Receives the broadcast.
# - Responds with compliance checks and contract actions.
# 👩‍💼 HR Agent
# - Receives the broadcast.
# - Responds with staffing and training plans.

# 🧑‍⚖️ Decision Agent
# - Collects all responses.
# - Integrates them into a Final Corporate Launch Plan.
# - In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.

In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.3 MB/s eta 0:00:00


In [3]:
import asyncio
from groq import Groq
from google.colab import userdata

# Load API key from Colab Secret
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"


# Base Agent Class
class Agent:
    def __init__(self, name, role):
        self.name = name
        self.role = role

    async def respond(self, message):
        prompt = f"""
        You are {self.name}, a {self.role}.

        Announcement:
        {message}

        Provide your department's action plan in a clear and professional way.
        Keep it natural and avoid symbols or decorative formatting.
        """

        response = await asyncio.to_thread(
            client.chat.completions.create,
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.6
        )

        return response.choices[0].message.content.strip()


# Coordinator Agent
class Coordinator:
    def __init__(self, message):
        self.message = message

    async def broadcast(self, agents):
        print("\nLaunch Announcement:")
        print(self.message)

        tasks = [agent.respond(self.message) for agent in agents.values()]
        results = await asyncio.gather(*tasks)

        return dict(zip(agents.keys(), results))


# Decision Agent
class DecisionAgent:
    async def generate_plan(self, responses):
        combined = "\n\n".join(
            [f"{name} Response:\n{resp}" for name, resp in responses.items()]
        )

        prompt = f"""
        You are a senior corporate strategist.

        Based on the following department responses, create a final product launch plan.

        Combine all inputs into one clear business plan.
        Keep it professional and human-like.

        {combined}
        """

        response = await asyncio.to_thread(
            client.chat.completions.create,
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.6
        )

        return response.choices[0].message.content.strip()


# Create Agents
marketing_agent = Agent(
    "Marketing Agent",
    "expert in marketing strategy, campaigns, and positioning"
)

finance_agent = Agent(
    "Finance Agent",
    "expert in budgeting and ROI forecasting"
)

operations_agent = Agent(
    "Operations Agent",
    "expert in production and supply chain management"
)

legal_agent = Agent(
    "Legal Agent",
    "expert in compliance and contracts"
)

hr_agent = Agent(
    "HR Agent",
    "expert in staffing and training"
)


# Main Function
async def main():
    announcement = "Product X launch in Q3, target market North America, budget 5 million dollars."

    agents = {
        "Marketing": marketing_agent,
        "Finance": finance_agent,
        "Operations": operations_agent,
        "Legal": legal_agent,
        "HR": hr_agent
    }

    coordinator = Coordinator(announcement)

    # Broadcast to all agents in parallel
    responses = await coordinator.broadcast(agents)

    # Print responses
    for name, response in responses.items():
        print(f"\n{name} Agent Response:")
        print(response)

    # Final decision
    decision_agent = DecisionAgent()
    final_plan = await decision_agent.generate_plan(responses)

    print("\nFinal Corporate Launch Plan:")
    print(final_plan)


# Run in Google Colab
await main()


Launch Announcement:
Product X launch in Q3, target market North America, budget 5 million dollars.

Marketing Agent Response:
As the Marketing Agent, I am excited to announce our department's action plan for the launch of Product X in Q3, targeting the North American market with a budget of 5 million dollars.

Our overall objective is to create awareness, generate interest, and drive sales of Product X in the North American market. To achieve this, we will focus on the following key strategies:

First, we will conduct market research to gain a deeper understanding of our target audience, their needs, and preferences. This will involve analyzing industry trends, competitor activity, and customer feedback to identify opportunities and challenges.

Next, we will develop a comprehensive marketing campaign that includes a mix of online and offline tactics. This will include social media marketing, email marketing, content marketing, search engine optimization, and paid advertising. We wil